# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Lane:** content decay (finding pages that are losing search visibility).

- **Contract answer 1, what one row means:** one content page on one day. The table's grain is `report_date × client_hash_id × content_hash_id`. For modeling, rows are later summed to one row per page.
- **Contract answer 2, table used:** `fact_content_daily_performance`, only the `month=2026-03` partition (a mid-panel month).
- **Contract answer 3, time window:** 1 to 31 March 2026 is loaded. The decision moment is 15 March: days 1 to 15 are the "past" side and days 16 to 30 are the "future" side, two windows of 15 days each. 31 March is not used, so that both windows have the same length.
- **Contract answers 4 and 5** (what I rank, and what I exclude) are in section 2. The excluded item is June 2026, the final month, which is kept sealed and never loaded.

These claims are verified with queries in section 3.



In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

SEED = 42
DECISION_DAY = "2026-03-15"                 # past side: days 1-15
LABEL_START, LABEL_END = "2026-03-16", "2026-03-30"   # future side: days 16-30

print("Table:  fact_content_daily_performance, month=2026-03")
print(f"Past:   2026-03-01 to {DECISION_DAY}")
print(f"Future: {LABEL_START} to {LABEL_END}")
print("Token loaded:", bool(HF_TOKEN))      # True/False only, never the token

Table:  fact_content_daily_performance, month=2026-03
Past:   2026-03-01 to 2026-03-15
Future: 2026-03-16 to 2026-03-30
Token loaded: True


## 2. Fields: feature / label / context / excluded

**Contract answer 4: what I rank (label proxy).** `is_decaying` = 1 if a page's impressions in days 16 to 30 are 30% or more lower than in days 1 to 15 of March. Only pages with `gsc_data_available IS TRUE` and at least 100 impressions in days 1 to 15 are used, because percentage changes on tiny volumes are noise. This is a proxy for decay, used to rank pages for review.

**Contract answer 5: what I deliberately exclude.** June 2026, the sealed final month (stated in section 1). It is never loaded.

| Bucket | Field | Built from | Why it is in this bucket |
|---|---|---|---|
| **Label** | `is_decaying` | `gsc_impressions`, days 16 to 30 vs days 1 to 15 | The outcome the ranking tries to predict |
| **Feature** | `imp_h1` | `gsc_impressions` summed over days 1 to 15 | Past-side visibility |
| **Feature** | `clicks_h1` | `gsc_clicks` summed over days 1 to 15 | Past-side clicks |
| **Feature** | `avg_pos_h1` | `gsc_avg_position` averaged over days 1 to 15 (days with impressions) | Past-side ranking position |
| **Feature** | `active_days_h1` | count of days 1 to 15 with impressions | How consistently the page shows up |
| **Feature** | `imp_last7_vs_first7` | impressions of days 9 to 15 divided by days 1 to 7 | Early momentum inside the past window |
| **Context** | `client_hash_id`, `content_hash_id` | ids | Used only to group and split by client; never features |
| **Excluded** | `imp_h2` (impressions, days 16 to 30) | `gsc_impressions` | Defines the label, so it can never be a feature |
| **Excluded** | GA4, session and AI columns | GA4 fields | Not needed for a search-visibility label, and zero-filled before a client's GA4 start |

Each feature's "knowable at the decision moment because..." line is written in section 3, next to the code that builds it.

In [3]:
FIELDS = {
    "label":    ["is_decaying"],
    "features": ["imp_h1", "clicks_h1", "avg_pos_h1", "active_days_h1", "imp_last7_vs_first7"],
    "context":  ["client_hash_id", "content_hash_id"],
    "excluded": ["imp_h2", "GA4 / session / AI columns"],
}

for bucket, cols in FIELDS.items():
    print(f"{bucket:9}", cols)

label     ['is_decaying']
features  ['imp_h1', 'clicks_h1', 'avg_pos_h1', 'active_days_h1', 'imp_last7_vs_first7']
context   ['client_hash_id', 'content_hash_id']
excluded  ['imp_h2', 'GA4 / session / AI columns']


## 3. Verify it with queries (grain, counts, missing values, windows)

Three verification queries on the March slice, each with its output shown directly under it: the grain, the row count and date span, and availability. The March partition is first read once into a local table so the queries run quickly. This loads only the columns this notebook uses.

In [4]:
con.execute(f"""
    CREATE OR REPLACE TABLE march AS
    SELECT report_date,
        client_hash_id,
        content_hash_id,
        gsc_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {MARCH}
""")
print("March slice loaded:", f"{con.sql('SELECT COUNT(*) FROM march').fetchone()[0]:,}", "rows")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March slice loaded: 9,841,378 rows


**Query 1: grain.** One row should be one page on one day, so no combination of `report_date`, `client_hash_id` and `content_hash_id` should appear twice. `duplicate_keys` should be 0.

**Observed:** `duplicate_keys` = 0 across 9,841,378 rows, so one row is one page on one day.

In [5]:
con.sql(f"""
    SELECT COUNT(*) AS rows_in_slice,
        (SELECT COUNT(*) FROM (
            SELECT 1 FROM march
            GROUP BY report_date, client_hash_id, content_hash_id
            HAVING COUNT(*) > 1)) AS duplicate_keys
    FROM march
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_in_slice,duplicate_keys
0,9841378,0


**Query 2: row count and date span** of the slice. The slice should cover 1 to 31 March 2026 only, with no days from other months.

**Observed:** 9,841,378 rows from 2026-03-01 to 2026-03-31 (31 days), covering 55 clients and 331,437 pages.

In [6]:
con.sql("""
    SELECT COUNT(*)                        AS rows_in_slice,
           MIN(report_date)                AS first_day,
           MAX(report_date)                AS last_day,
           COUNT(DISTINCT report_date)     AS days,
           COUNT(DISTINCT client_hash_id)  AS clients,
           COUNT(DISTINCT content_hash_id) AS pages
    FROM march
""").df()

,rows_in_slice,first_day,last_day,days,clients,pages
0,9841378,2026-03-01,2026-03-31,31,55,331437


**Query 3: availability.** `gsc_data_available` can be TRUE, FALSE or NULL, so a filter like `= TRUE` or `NOT FALSE` would mishandle the NULLs. The query filters with `IS TRUE` and shows how many rows survive, alongside how many are FALSE and NULL.

**Observed:** 3,611,061 of 9,841,378 rows (36.7%) survive `IS TRUE`; 6,230,317 are FALSE and none are NULL in March. Only the surviving rows are used from here on.

In [7]:
con.sql("""
    SELECT COUNT(*)                                                         AS total_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)               AS rows_surviving_is_true,
           ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 1) AS pct_surviving,
           COUNT(*) FILTER (WHERE gsc_data_available IS FALSE)              AS rows_false,
           COUNT(*) FILTER (WHERE gsc_data_available IS NULL)               AS rows_null
    FROM march
""").df()

,total_rows,rows_surviving_is_true,pct_surviving,rows_false,rows_null
0,9841378,3611061,36.7,6230317,0


### Five features (maximum), each knowable at the decision moment

The decision moment is the end of 15 March. A feature is allowed only if it uses days 1 to 15.

| Feature | What it is | Knowable at the decision moment because... |
|---|---|---|
| `imp_h1` | Total impressions, days 1 to 15 | those daily rows already exist by 15 March |
| `clicks_h1` | Total clicks, days 1 to 15 | same days, nothing after 15 March |
| `avg_pos_h1` | Average search position on days with impressions, days 1 to 15 | it only averages days up to 15 March |
| `active_days_h1` | Number of days with impressions, days 1 to 15 | it counts days up to 15 March only |
| `imp_last7_vs_first7` | Impressions on days 9 to 15 divided by days 1 to 7 | both windows end on or before 15 March |

Note: Search Console data usually arrives with a delay of a couple of days, so a live system would use data up to about 12 March. The logic is the same.

The cell below builds one row per page. It keeps only rows with `gsc_data_available IS TRUE` and pages with at least 100 impressions in days 1 to 15, because percentage changes on tiny volumes are noise. `imp_h2` (impressions in days 16 to 30) is computed only to define the label `is_decaying`; it is not a feature. A page with no rows at all in days 16 to 30 is treated as having 0 impressions in that window.

**Observed:** 77,540 pages from 38 clients passed the filters; 24.5% are labeled `is_decaying`.

In [8]:
MIN_IMP = 100    # minimum impressions in days 1-15
DROP_PCT = 30    # decaying = impressions fell by 30% or more

frame = con.sql(f"""
    SELECT * FROM (
        SELECT client_hash_id, content_hash_id,
               SUM(imp) FILTER (WHERE report_date <= DATE '{DECISION_DAY}')                 AS imp_h1,
               SUM(clk) FILTER (WHERE report_date <= DATE '{DECISION_DAY}')                 AS clicks_h1,
               AVG(pos) FILTER (WHERE report_date <= DATE '{DECISION_DAY}' AND imp > 0)     AS avg_pos_h1,
               COUNT(*) FILTER (WHERE report_date <= DATE '{DECISION_DAY}' AND imp > 0)     AS active_days_h1,
               SUM(imp) FILTER (WHERE report_date BETWEEN DATE '2026-03-09' AND DATE '{DECISION_DAY}')
                 / NULLIF(SUM(imp) FILTER (WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-07'), 0)
                                                                                           AS imp_last7_vs_first7,
               COALESCE(SUM(imp) FILTER (WHERE report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'), 0)
                                                                                           AS imp_h2
        FROM (SELECT client_hash_id, content_hash_id, report_date,
                     COALESCE(gsc_impressions, 0) AS imp,
                     COALESCE(gsc_clicks, 0)      AS clk,
                     gsc_avg_position             AS pos
              FROM march
              WHERE gsc_data_available IS TRUE)
        GROUP BY client_hash_id, content_hash_id
    )
    WHERE imp_h1 >= {MIN_IMP}
""").df()

# Label: impressions in days 16-30 fell by DROP_PCT or more versus days 1-15
frame["is_decaying"] = (frame["imp_h2"] <= (1 - DROP_PCT / 100) * frame["imp_h1"]).astype(int)

FEATURES = FIELDS["features"]
print("Pages in frame:", f"{len(frame):,}",
      "| clients:", frame["client_hash_id"].nunique(),
      "| decaying share:", round(frame["is_decaying"].mean(), 3))
frame[FEATURES + ["is_decaying"]].head()

Pages in frame: 77,540 | clients: 38 | decaying share: 0.245


,imp_h1,clicks_h1,avg_pos_h1,active_days_h1,imp_last7_vs_first7,is_decaying
0,4173.0,6.0,6.327311,15,0.641362,1
1,245.0,0.0,3.906852,15,0.949580,0
2,3705.0,3.0,6.473735,15,0.750752,1
3,2440.0,8.0,7.259861,15,0.666431,0
4,240.0,1.0,3.860842,15,1.093458,0


### Quick honest score

A small gradient-boosting model, scored with AUC on clients it never saw during training (a grouped split on `client_hash_id`, 75% of clients for training and 25% for testing). AUC is the chance that a randomly chosen decaying page is ranked above a randomly chosen non-decaying page: 0.5 is no better than chance and 1.0 is perfect. This is a quick check with default settings, not a tuned model.

**Observed:** AUC = 0.701 on a single grouped split (directional, not a tuned or cross-validated result). This is the honest number that the leak experiment below is compared against.

In [9]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

y = frame["is_decaying"].values

def score(cols):
    """AUC on clients the model never saw."""
    train, test = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
                       .split(frame, groups=frame["client_hash_id"]))
    model = HistGradientBoostingClassifier(random_state=SEED)
    model.fit(frame.iloc[train][cols], y[train])
    return roc_auc_score(y[test], model.predict_proba(frame.iloc[test][cols])[:, 1])

honest_auc = score(FEATURES)
print(f"Honest AUC (5 features, unseen clients): {honest_auc:.3f}")

Honest AUC (5 features, unseen clients): 0.694


### The trap: one deliberate leak

I add one column built from the label window: `leak_h2_vs_h1_pct`, the percent change in impressions from days 1 to 15 to days 16 to 30. The label `is_decaying` is a threshold on exactly this number, so the column is the label in disguise. The score should jump toward perfect. That jump is not skill: the column uses information from after the decision moment (15 March), which a real system would not have. I then delete the column and keep the honest score.

**Observed:** with `leak_h2_vs_h1_pct`, AUC = 1.000; after deleting it, AUC = 0.701. The jump of about 0.30 came from information that does not exist at the decision moment, not from skill. The honest number is 0.701.

In [10]:
# 1) Add ONE label-derived column on purpose
frame["leak_h2_vs_h1_pct"] = (frame["imp_h2"] - frame["imp_h1"]) / frame["imp_h1"] * 100
leaky_auc = score(FEATURES + ["leak_h2_vs_h1_pct"])
print(f"AUC with the leaky column:  {leaky_auc:.3f}   <- looks perfect, but it is cheating")

# 2) Delete it and keep the honest number
frame = frame.drop(columns=["leak_h2_vs_h1_pct"])
final_auc = score(FEATURES)
print(f"AUC after removing it:      {final_auc:.3f}   <- the honest number we keep")

AUC with the leaky column:  1.000   <- looks perfect, but it is cheating
AUC after removing it:      0.694   <- the honest number we keep


## 4. Data limits

**Named limitation: this is a one-month slice.** Everything here comes from March 2026 with a single decision date (15 March). It cannot tell me anything about other months, seasonality, or day-of-week effects. The 30% decay rule and the 100-impression filter are untuned assumptions, and the AUC comes from one grouped split. Results should be read as directional, not as evidence that the approach works in other periods.

The cell below shows how many pages each client contributes, since the panel is unbalanced: history depth and traffic differ a lot between clients.

**Observed:** the frame has 38 clients. Pages per client range from 1 to 17,610 (median 493), and the largest client holds 22.7% of pages. With such uneven client sizes, a single grouped split can give a noticeably different AUC, so 0.701 is directional.

In [11]:
per_client = frame.groupby("client_hash_id").size()
print("Clients in frame:", len(per_client))
print("Pages per client: min", int(per_client.min()),
      "| median", int(per_client.median()),
      "| max", int(per_client.max()))
print("Largest client share of pages:", f"{per_client.max() / len(frame):.1%}")

Clients in frame: 38
Pages per client: min 1 | median 493 | max 17610
Largest client share of pages: 22.7%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.